In [12]:
import pandas as pd
import ast

# Load the datasets
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# Merge them based on the movie title
movies = movies.merge(credits, on='title')

In [13]:
# Keep only the columns we need for content filtering
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Helper function to extract names from the JSON dictionaries
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

# Apply the function to clean up the columns
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# For cast, we only want the top 3 actors
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter+=1
    return L
movies['cast'] = movies['cast'].apply(convert3)

# For crew, we only want the Director
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L
movies['crew'] = movies['crew'].apply(fetch_director)

In [14]:
# Remove spaces between words in lists
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

# Convert overview string to a list of words
movies['overview'] = movies['overview'].apply(lambda x:x.split() if type(x)==str else [])

# Create the master tags column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Create a clean final dataframe
new_df = movies[['movie_id', 'title', 'tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\adity\AppData\Local\Temp\ipykernel_22040\2402600218.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
C:\Users\adity\AppData\Local\Temp\ipykernel_22040\2402600218.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [15]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

# Convert text to a matrix of vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

from sklearn.metrics.pairwise import cosine_similarity
# Compare every movie vector to every other movie vector
similarity = cosine_similarity(vectors)

In [16]:
import pickle

# Save the final movie list dictionary and similarity matrix to disk
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

In [17]:
import pandas as pd

# Load the Bollywood dataset
bolly_df = pd.read_csv('BollywoodMovieDetail.csv')

# Print out the exact column names
print(bolly_df.columns.tolist())

# Show the first two rows to see how the data is formatted
bolly_df.head(2)

['imdbId', 'title', 'releaseYear', 'releaseDate', 'genre', 'writers', 'actors', 'directors', 'sequel', 'hitFlop']


,imdbId,title,releaseYear,releaseDate,genre,writers,actors,directors,sequel,hitFlop
0,tt0118578,Albela,2001,20 Apr 2001,Romance,Honey Irani (screenplay) | Honey Irani (story)...,Govinda | Aishwarya Rai Bachchan | Jackie Shro...,Deepak Sareen,0.0,2
1,tt0169102,Lagaan: Once Upon a Time in India,2001,08 May 2002,Adventure | Drama | Musical,Ashutosh Gowariker (story) | Ashutosh Gowarike...,Aamir Khan | Gracy Singh | Rachel Shelley | Pa...,Ashutosh Gowariker,0.0,6


In [18]:
import pandas as pd
import ast

# 1. Load both datasets
tmdb_movies = pd.read_csv('tmdb_5000_movies.csv')
tmdb_credits = pd.read_csv('tmdb_5000_credits.csv')

# Merge TMDB data together (standard recommender setup)
movies = tmdb_movies.merge(tmdb_credits, on='title')

# Load your Bollywood data
bolly_df = pd.read_csv('BollywoodMovieDetail.csv')

# 2. Harmonize Bollywood columns to match TMDB structure
# Let's create a combined 'tags' text field for Bollywood using genre, actors, directors, and plot/writers
bolly_df['genres'] = bolly_df['genre'].fillna('')
bolly_df['cast'] = bolly_df['actors'].fillna('')
bolly_df['crew'] = bolly_df['directors'].fillna('')

# Create a combined tag column for Bollywood
bolly_df['tags'] = bolly_df['genres'] + " " + bolly_df['cast'] + " " + bolly_df['crew']

# Select and rename columns so they match what your model expects
bolly_processed = pd.DataFrame({
    'movie_id': bolly_df['imdbId'],
    'title': bolly_df['title'],
    'tags': bolly_df['tags'],
    'release_date': bolly_df['releaseYear'], # Keeping release year
})

# For TMDB, let's select its core columns too (assuming you have a cleaning function for tags already)
# If your TMDB 'new_df' already exists, you can concatenate it directly. 
# Let's assume standard TMDB processing:
movies['tags'] = movies['overview'].fillna('') + " " + movies['genres'].fillna('') # (adjust if your custom tag builder differs)

tmdb_processed = pd.DataFrame({
    'movie_id': movies['movie_id'],
    'title': movies['title'],
    'tags': movies['tags'],
    'release_date': movies['release_date'] if 'release_date' in movies.columns else "2000"
})

# 3. Combine both Hollywood and Bollywood into a single master dataframe!
master_df = pd.concat([tmdb_processed, bolly_processed], ignore_index=True)

# Drop any rows with missing titles or tags
master_df.dropna(subset=['title', 'tags'], inplace=True)
master_df.reset_index(drop=True, inplace=True)

print(f"Total Combined Movies: {master_df.shape[0]}")

Total Combined Movies: 6093


In [19]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

# 1. Initialize CountVectorizer to convert movie tags into numerical vectors
cv = CountVectorizer(max_features=5000, stop_words='english')

# Convert the 'tags' column of your master_df
vectors = cv.fit_transform(master_df['tags']).toarray()

# 2. Calculate the cosine similarity matrix for all movies (Hollywood + Bollywood)
similarity = cosine_similarity(vectors)

# 3. Save the updated dataframe and similarity matrix into pickle files
# We pass master_df as a dictionary so it matches how your app.py expects it
pickle.dump(master_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Successfully generated and saved new pickle files with Bollywood movies!")

Successfully generated and saved new pickle files with Bollywood movies!


In [20]:
# Check if a movie exists in your combined dataframe
search_term = "Spider-Man"
result = master_df[master_df['title'].str.contains(search_term, case=False, na=False)]
print(result[['title', 'movie_id']])

                        title movie_id
5                Spider-Man 3      559
20     The Amazing Spider-Man     1930
30               Spider-Man 2      558
38   The Amazing Spider-Man 2   102382
159                Spider-Man      557


In [21]:
import pandas as pd
import ast

# 1. Load TMDB datasets properly
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')
movies = movies.merge(credits, on='title')

# Helper function to extract lists from JSON strings in TMDB
def convert(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            L.append(i['name'])
    except:
        pass
    return L

def convert3(text):
    L = []
    counter = 0
    try:
        for i in ast.literal_eval(text):
            if counter != 3:
                L.append(i['name'])
                counter += 1
            else:
                break
    except:
        pass
    return L

def fetch_director(text):
    L = []
    try:
        for i in ast.literal_eval(text):
            if i['job'] == 'Director':
                L.append(i['name'])
                break
    except:
        pass
    return L

# Clean TMDB columns
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert3)
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].fillna('').apply(lambda x: x.split())

# Format TMDB lists into strings
for col in ['genres', 'keywords', 'cast', 'crew']:
    movies[col] = movies[col].apply(lambda x: [i.replace(" ", "") for i in x])

movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x))

# Create clean TMDB dataframe
tmdb_processed = pd.DataFrame({
    'movie_id': movies['movie_id'],
    'title': movies['title'],
    'tags': movies['tags'],
    'release_date': movies['release_date'].astype(str).str.extract('(\d{4})')[0].fillna('2000')
})

# 2. Load and process Bollywood data
bolly_df = pd.read_csv('BollywoodMovieDetail.csv')
bolly_df['tags'] = bolly_df['genre'].fillna('') + " " + bolly_df['actors'].fillna('') + " " + bolly_df['directors'].fillna('')

bolly_processed = pd.DataFrame({
    'movie_id': bolly_df['imdbId'],
    'title': bolly_df['title'],
    'tags': bolly_df['tags'],
    'release_date': bolly_df['releaseYear'].astype(str)
})

# 3. Combine both into master_df
master_df = pd.concat([tmdb_processed, bolly_processed], ignore_index=True)
master_df.dropna(subset=['title', 'tags'], inplace=True)
master_df.reset_index(drop=True, inplace=True)

print("Master dataset ready with rows:", master_df.shape[0])

<>:63: SyntaxWarning: invalid escape sequence '\d'
<>:63: SyntaxWarning: invalid escape sequence '\d'
C:\Users\adity\AppData\Local\Temp\ipykernel_22040\3287603246.py:63: SyntaxWarning: invalid escape sequence '\d'
  'release_date': movies['release_date'].astype(str).str.extract('(\d{4})')[0].fillna('2000')


Master dataset ready with rows: 6093


In [22]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(master_df['tags']).toarray()

similarity = cosine_similarity(vectors)

# Overwrite pickles
pickle.dump(master_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Pickles updated successfully!")

Pickles updated successfully!
